## 4.1 Transformer Block
一个 decoder-only LLM 的主体，其实就是把同一种 Block 重复堆很多层。
只要真正理解了一个 Block 的 forward 逻辑，后面 12 层、24 层、48 层本质上都只是“复制粘贴加参数量”。
那么，为什么能一直堆呢？为什么不会堆深以后直接训崩呢？
- 残差连接
  - 梯度永远不会消失
  - 网络退化（退化不是过拟合。过拟合是训练集很好、测试集变差；退化是连训练集都练不动）
    - 因为一个很深的 plain network，理论上当然可以学出“至少不比浅层差”的函数； 但优化器未必能轻松找到这条路。
    - 如果新增的几层其实没什么用，最合理的行为应该是“学成恒等映射”。
- Pre-Norm
| 结构        | 公式                      | 优点         | 缺点        | 现代使用情况                 |
| --------- | ----------------------- | ---------- | --------- | ---------------------- |
| Post-Norm | `y = Norm(x + Sublayer(x))` | 结构直观       | 深层训练更容易不稳 | 原始 Transformer 常见      |
| Pre-Norm  | `y = x + Sublayer(Norm(x))` | 梯度流更顺、深层更稳 | 理解上稍绕一点   | 现代 decoder-only LLM 主流 |



In [1]:
import sys
from pathlib import Path

ROOT = Path("..").absolute()
sys.path.append(str(ROOT))

import torch
import torch.nn as nn
import import_ipynb

# 第二章：基础层
from transformer.transformer import MyEmbedding, TransformerLinear

# 第三章：注意力、位置编码、FFN、Norm
from attention.attention import CausalMultiHeadSelfAttention, RotaryPositionalEmbedding, SwiGLU, RMSNorm, LayerNorm

In [2]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        max_seq_len: int,
        rope_theta: float = 10000.0,
        device="cuda",
        dtype=None,
        attn_module: nn.Module | None = None,
        norm_type: str = "rmsnorm",
        ffn_type: str = "swiglu"
    ):
        super().__init__()
        # 1) Norm 选择
        if norm_type == "rmsnorm":
            self.ln1 = RMSNorm(d_model, device=device, dtype=dtype)
            self.ln2 = RMSNorm(d_model, device=device, dtype=dtype)
        elif norm_type == "layernorm":
            self.ln1 = LayerNorm(d_model, device=device, dtype=dtype)
            self.ln2 = LayerNorm(d_model, device=device, dtype=dtype)
        else:
            raise ValueError(f"Unsupported norm type: {norm_type}")
        
        # 2) 注意力模块选择
        if attn_module is not None:
            self.attn = attn_module
        else:
            self.attn = CausalMultiHeadSelfAttention(
                d_model=d_model,
                num_heads=num_heads,
                max_seq_len=max_seq_len,
                rope_theta=rope_theta,
                device=device,
                dtype=dtype
            )
        
        # 3) FFN 模块选择
        if ffn_type == "swiglu":
            self.ffn = SwiGLU(d_model, d_ff, device=device, dtype=dtype)
        elif ffn_type == "linear":
            self.ffn = TransformerLinear(d_model, d_ff, device=device, dtype=dtype)
        else:
            raise ValueError(f"Unsupported FFN type: {ffn_type}")
        
    def forward(self, x:torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        # pre-norm
        normed_x1 = self.ln1(x)
        # 注意力子层
        attn_out = self.attn(normed_x1, token_positions=token_positions)
        # 残差连接
        x = x + attn_out

        # FFN 子层
        normed_x2 = self.ln2(x)
        x = x + self.ffn(normed_x2)

        return x

## 4.2 TransformerLM
可以把 `TransformerLM` 理解成一个四段式工厂：
1. **入口**：Token Embedding
2. **加工区**：N 个 Transformer Blocks
3. **终检**：Final Norm
4. **出口**：LM Head，产出 logits
```
token_ids           [B, S]
 -> token_embedding [B, S, D]
 -> N x Block       [B, S, D]
 -> final_norm      [B, S, D]
 -> lm_head         [B, S, V]
```


In [3]:
class TransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int,
        d_model: int,
        num_layers: int,
        num_heads: int,
        d_ff: int,
        rope_theta: float,
        device=None,
        dtype=None,
        norm_type: str = "rmsnorm",
        ffn_type: str = "swiglu",
        tie_embeddings: bool = True, # 是否共享输入输出嵌入矩阵
        block_factory=None, # 可选的 TransformerBlock 工厂函数，可以替换GQA/MQA/MLA等
    ):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.vocab_size = vocab_size
    
        # 1) 输入嵌入层
        self.token_embedding = MyEmbedding(vocab_size, d_model, device=device, dtype=dtype)

        # 2) Transformer Block 堆叠
        if block_factory is None:
            self.layers = nn.ModuleList([       # 普通List没有注册
                TransformerBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    d_ff=d_ff,
                    max_seq_len=max_seq_len,
                    rope_theta=rope_theta,
                    device=device,
                    dtype=dtype,
                    norm_type=norm_type,
                    ffn_type=ffn_type
                )
                for _ in range(num_layers)
            ])
        else:
            self.layers = nn.ModuleList([
                block_factory(layer_idx=i) for i in range(num_layers)
            ])

        # 3) Final Norm
        # Block 里面都已经有 Norm 了，为什么最后还要一个 `ln_final`？
        # 因为 Block 里的 Norm 是为了每个子层稳定训练；而最终输出到 `lm_head` 之前，再做一次统一归一化，能让最后的隐藏状态尺度更整齐。
        # - Block 内的 Norm：服务于深层优化
        # - Final Norm：服务于最终输出投影
        if norm_type == "rmsnorm":
            self.ln_final = RMSNorm(d_model, device=device, dtype=dtype)
        elif norm_type == "layernorm":
            self.ln_final = LayerNorm(d_model, device=device, dtype=dtype)
        else:
            raise ValueError(f"Unsupported norm type: {norm_type}")
        
        # 4) LM Head
        # [B, S, D] -> [B, S, V]
        # 不是直接输出 token，而是对词表里每个 token 给一个 logit 分数。
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False, device=device, dtype=dtype)

        # optional: weight tying
        # 如果 `tie_embeddings` 为 True，则将 `lm_head` 的权重矩阵与 `token_embedding` 的权重矩阵共享
        # 这意味着输入嵌入和输出投影使用同一套参数，通常可以减少模型参数量并提升性能。
        if tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids:torch.Tensor) -> torch.Tensor:
        """
        input_ids: [B, S]
        output: [B, S, V]
        """
        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.max_seq_len, f"Sequence length {seq_len} exceeds model's max_seq_len {self.max_seq_len}"

        # RoPE
        token_positions = torch.arange(
            seq_len, device=input_ids.device
        ).unsqueeze(0).expand(batch_size, seq_len) # [B, S]
        # 注意：RoPE 的位置编码是在注意力模块内部实现的，所以这里不需要显式地添加位置编码到输入嵌入中。

        # 1) 输入嵌入
        x = self.token_embedding(input_ids) # [B, S, D]

        # 2) Transformer Block 堆叠
        for block in self.layers:
            x = block(x, token_positions=token_positions) # [B, S, D]

        # 3) Final Norm
        x = self.ln_final(x) # [B, S, D]

        # 4) Vocabulary projection
        logits = self.lm_head(x) # [B, S, V]
        return logits

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerLM(
    vocab_size=5000,
    max_seq_len=128,
    d_model=256,
    num_layers=4,
    num_heads=8,
    d_ff=768,          # 约等于 3x，也可以改成按 8/3 规则取整
    rope_theta=10000.0,
    device=device,
    dtype=torch.float32,
).to(device)

token_ids = torch.randint(0, 5000, (2, 16), device=device)
logits = model(token_ids)

print("token_ids shape:", token_ids.shape)
print("logits shape:", logits.shape)

token_ids shape: torch.Size([2, 16])
logits shape: torch.Size([2, 16, 5000])


: 

## 4.3 现代化升级
| 优先级 | 建议项                              | 为什么            |
| --- | -------------------------------- | -------------- |
| 最高  | MHA / GQA / MQA 统一实现             | 改动小，收益大，真实工业常用 |
| 很高  | QK-Norm                          | 实现不重，现代报告高频出现  |
| 中等  | Local / Sliding Window Attention | 更像长上下文工程优化     |
| 低   | MLA / MoE                         | 很强，但架构改动明显更大   |
